Gold notebook: RFM stands for recency, frequency, and monetary. This way we can see when they bought, how much they bought, and the value (quantity x price)

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "ltv.duckdb"

cols = ["Invoice", "Quantity", "InvoiceDate", "Price", "Customer_ID", "Country",
        "Customer_ID_Exists", "Cancelled_Order", "Return_Adjustment"]

with duckdb.connect(str(DB_PATH)) as con:
    silver = con.sql(f"SELECT {', '.join(cols)} FROM silver").df()

print(f"Silver rows loaded: {len(silver):,}")

Silver rows loaded: 1,067,371


In [2]:
with duckdb.connect(str(DB_PATH)) as con:
    silver = con.sql("""
        SELECT Invoice, InvoiceDate, Customer_ID, Quantity, Price,
               Customer_ID_Exists, Cancelled_Order, Return_Adjustment
        FROM silver
    """).df()

sales = silver[
    silver["Customer_ID_Exists"]
    & ~silver["Cancelled_Order"] #~ means NOT, so this is filtering out cancelled orders
    & ~silver["Return_Adjustment"] #~ means NOT, so this is filtering out returns/adjustments
    & (silver["Quantity"] > 0)
    & (silver["Price"] > 0)
].copy()
del silver

sales["line_total"] = sales["Quantity"] * sales["Price"]
print(f"Usable sales rows: {len(sales):,}")
print(f"Customers: {sales['Customer_ID'].nunique():,}")

Usable sales rows: 805,549
Customers: 5,878


In [ ]:
sales["InvoiceDate"] = pd.to_datetime(sales["InvoiceDate"])
snapshot = sales["InvoiceDate"].max() + pd.Timedelta(days=1)   # "today" for recency

gold = sales.groupby("Customer_ID").agg(
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max"),
    frequency=("Invoice", "nunique"),
    monetary=("line_total", "sum"),
).reset_index()

gold["recency_days"] = (snapshot - gold["last_purchase"]).dt.days
gold["tenure_days"] = (snapshot - gold["first_purchase"]).dt.days
gold["first_purchase_month"] = gold["first_purchase"].dt.to_period("M").astype(str)

gold["first_purchase"] = gold["first_purchase"].dt.date
gold["last_purchase"] = gold["last_purchase"].dt.date

print(f"Snapshot date: {snapshot.date()}")
print(gold.describe().round(1))
gold.head()

Snapshot date: 2011-12-10
       Customer_ID  frequency  monetary  recency_days  tenure_days
count       5878.0     5878.0    5878.0        5878.0       5878.0
mean       15315.3        6.3    3018.6         201.9        475.3
std         1715.6       13.0   14737.7         209.4        223.1
min        12346.0        1.0       3.0           1.0          1.0
25%        13833.2        1.0     348.8          26.0        314.0
50%        15314.5        3.0     898.9          96.0        531.0
75%        16797.8        7.0    2307.1         380.0        669.0
max        18287.0      398.0  608821.6         739.0        739.0


,Customer_ID,first_purchase,last_purchase,frequency,monetary,recency_days,tenure_days,first_purchase_month
0,12346,2009-12-14,2011-01-18,12,77556.46,326,726,2009-12
1,12347,2010-10-31,2011-12-07,8,5633.32,3,405,2010-10
2,12348,2010-09-27,2011-09-25,5,2019.40,76,439,2010-09
3,12349,2010-04-29,2011-11-21,4,4428.69,19,590,2010-04
4,12350,2011-02-02,2011-02-02,1,334.40,311,311,2011-02
